# 003：固定预算下的无进展识别与收尾策略（A0/B/C）

A0共同修订基线 → B无进展抑制 → C软收尾。先6题探索（ABC顺序平衡）再24题配对；heldout冻结。复用001已修运行逻辑（R01–R19）。

本机只编辑校验；GPU运行在Kaggle Linux。


## 0. 路径与版本记录（先运行，R01）

只用标准库+distribution元数据；安装后重启kernel再执行。


In [ ]:
import hashlib
import json
import platform
import sys
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_SRC = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# 本项目 agent/ 在 Kaggle 中的位置：优先 /kaggle/working/agent（用户上传），其次相对路径 agent/（repo 模式）
CANDIDATES = [Path('/kaggle/working/agent'), Path('agent'), Path('/kaggle/working/agent_001')]
PROJECT_AGENT_SRC = next((p for p in CANDIDATES if (p / 'agent.yaml').exists()), None)
# R04：执行目录不得复用固定 agent_run 覆盖；此处仅记录来源，实际执行目录在 cell c-2 按 RUN_ID 创建。
AGENT_DIR = None
print(f'python={sys.version.split()[0]} platform={platform.platform()}')
print(f'DATA_DIR exists={DATA_DIR.exists()} MODEL_PATH exists={MODEL_PATH.exists()}')
print(f'PROJECT_AGENT_SRC={PROJECT_AGENT_SRC}')
# R01：安装前只用标准库 + distribution 元数据，不导入 torch / swegemma / vllm 等将被重装的栈。
# invalidate_caches() 不替换 sys.modules 中已加载对象；若 kernel 已加载目标模块，须安装后重启再执行。
import importlib.metadata as _md
TARGET_MODS = ['torch', 'swegemma', 'adk_submission', 'adk_eval_core', 'vllm', 'litellm', 'google.adk']
LOADED = [m for m in TARGET_MODS if m in sys.modules]
def _mpath(m):
    try:
        return getattr(sys.modules[m], '__file__', '?')
    except Exception:
        return '?'
CLEAN_KERNEL = (len(LOADED) == 0)
print(f'CLEAN_KERNEL={CLEAN_KERNEL} preloaded={LOADED}')
print(f'preloaded_paths={[(m, _mpath(m)) for m in LOADED]}')
if not CLEAN_KERNEL:
    print('WARNING(R01)：当前 kernel 已加载将被重装的模块；请完成安装 cell 后重启 kernel 再进入执行阶段，不要循环重装代替重启。')
for dist in ['swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm', 'torch']:
    try:
        print(f'{dist} dist-version={_md.version(dist)}')
    except Exception as e:
        print(f'{dist} NOT installed yet ({e})')
print('注：__import__(\'google.adk\') 返回顶层 google，不能据此打印 ADK 版本；以 importlib.metadata 为准 (R02)。')


## 1. 环境安装（Kaggle Linux only，R02）

版本化staging；只装清单内wheel；记录SHA-256。


In [ ]:
import glob
import importlib
import os
import subprocess
import sys
from pathlib import Path

assert sys.platform.startswith('linux'), '此单元格只在 Kaggle Linux GPU Notebook 运行，Mac 上跳过'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass

# R02：不用复用 /tmp/wheelhouse（旧链接/旧包会混入；官方 container_setup 解包缓存只按文件名复用）。
# 以来源内容清单建新 staging，只装本清单内 wheel；源空、悬空链接、同 distribution 多版本均报错。
import hashlib
src_wheels = sorted(WHEELHOUSE_SRC.glob('*.whl'))
assert len(src_wheels) > 0, f'wheelhouse 源为空：{WHEELHOUSE_SRC}（确认 fork 继承了 dataset 输入）'
manifest = []
_seen_dist = {}
for w in src_wheels:
    if 'cutlass' in w.name.lower():
        continue
    _dist = w.name.split('-')[0].replace('_', '-').lower()
    assert _dist not in _seen_dist, f'R02：同一distribution多版本 {_dist}: {_seen_dist[_dist]} vs {w.name}'
    _seen_dist[_dist] = w.name
    h = hashlib.sha256(w.read_bytes()).hexdigest()
    manifest.append((w.name, w.stat().st_size, h))
print(f'wheelhouse src count={len(manifest)}')
for name, size, h in manifest:
    print(f'  {name} size={size} sha256={h[:16]}…')
stage = Path('/tmp/wheelhouse-003-v1')
assert not stage.exists() or not any(stage.iterdir()), f'R02：staging 已有旧内容 {stage}，请换新干净目录执行，不要沿用旧缓存判断新环境'
stage.mkdir(parents=True, exist_ok=True)
for w in src_wheels:
    if 'cutlass' in w.name.lower():
        continue
    target_name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = stage / target_name
    assert not target.exists() and not target.is_symlink(), f'staging 冲突：{target}'
    assert w.exists() and not w.is_symlink() or w.resolve().exists(), f'悬空链接：{w}'
    os.symlink(w, target)
wheels = sorted(str(w) for w in stage.glob('*.whl'))
assert len(wheels) == len(manifest), 'staging 清单与来源不一致，中断'
print(f'Installing {len(wheels)} wheels from {stage}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
import json as _js
(WORKING_DIR / 'wheel_manifest_003.json').write_text(_js.dumps([{'name': n, 'size': s, 'sha256': h} for n, s, h in manifest], indent=2), encoding='utf-8')
print('Wheelhouse installation complete. 安装后请重启 kernel 再进入执行阶段 (R01)；执行阶段用 importlib.metadata 核对实际版本。')


## 2. 实验清单与变体校验（R03/R04/R06）

核对experiment_manifest、任务指纹、同组不跨集合；三组采样/预算一致、仅提示词不同；RUN_ID隔离。


In [ ]:
import hashlib
import json
import shutil
import yaml
from pathlib import Path
yaml.SafeLoader.add_constructor('!include', lambda l, n: l.construct_scalar(n))

# 003不依赖001 agent/上传；变体来源在下方VARIANT_SRC确定。
RUN_ID = 'run-003-explore'
VARIANT = 'a0'
assert VARIANT in ('a0', 'b', 'c'), f'未知VARIANT={VARIANT}'
REPEAT_ID = 'r1'
# R04：变体配置来自 research/003/variants/<VARIANT>/agent（权威）；001 agent/ 仅历史参照，不作本轮输入。
VCANDS = [Path(f'/kaggle/working/research/003/variants/{VARIANT}/agent'), Path(f'research/003/variants/{VARIANT}/agent')]
VARIANT_SRC = next((p for p in VCANDS if (p / 'agent.yaml').exists()), None)
assert VARIANT_SRC is not None, f'未找到变体配置 variants/{VARIANT}/agent（需上传 research/003）'
RUN_ID = 'run-003-explore'
RESULTS_ROOT = WORKING_DIR / 'results' / '003' / RUN_ID / VARIANT / REPEAT_ID
assert not RESULTS_ROOT.exists(), f'R04：{RESULTS_ROOT} 已存在，请换新 RUN_ID/REPEAT_ID（重跑用新repeat目录）'
AGENT_DIR = WORKING_DIR / 'agent_exec' / RUN_ID / VARIANT / REPEAT_ID
assert VARIANT_SRC.resolve() != AGENT_DIR.resolve(), '配置源与执行目录不可相同 (R04)'
assert AGENT_DIR.resolve() not in VARIANT_SRC.resolve().parents and VARIANT_SRC.resolve() not in AGENT_DIR.resolve().parents, '配置源与执行目录不可互为祖先 (R04)'
AGENT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(VARIANT_SRC, AGENT_DIR)
print(f'VARIANT={VARIANT} VARIANT_SRC={VARIANT_SRC}')
print(f'AGENT_DIR={AGENT_DIR}')
print(f'RESULTS_ROOT={RESULTS_ROOT} (先写 manifest 再开始，归档不依赖整批成功结束)')
# 三组采样/预算一致、仅提示词不同；与执行快照指纹核对 (R06)
import hashlib as _hl3
_cfgs = {}
for _v in ['a0', 'b', 'c']:
    _b = VARIANT_SRC.parent.parent / _v / 'agent'
    if not (_b / 'agent.yaml').exists():
        print(f'变体 {_v} 不在本地上传范围，跳过一致性核对（Kaggle上应三组齐全）')
        continue
    _s = (_b / 'configs' / 'sampling.yaml').read_text(encoding='utf-8')
    _e = (_b / 'eval_config.yaml').read_text(encoding='utf-8')
    _cfgs[_v] = (_hl3.sha256(_s.encode()).hexdigest(), _hl3.sha256(_e.encode()).hexdigest())
if len(_cfgs) > 1:
    assert len(set(v[0] for v in _cfgs.values())) == 1 and len(set(v[1] for v in _cfgs.values())) == 1, f'R04/R06：三组采样/预算必须一致：{_cfgs}'
    print(f'variant sampling/eval consistent across {sorted(_cfgs)}；差异仅system.md增量 (A0/B/C)')
print(f'执行配置指纹 agent={_hl3.sha256((AGENT_DIR / "agent.yaml").read_bytes()).hexdigest()[:16]} system={_hl3.sha256((AGENT_DIR / "prompts" / "system.md").read_bytes()).hexdigest()[:16]}')

# 数据版本（与 docs/plan/README 指纹比对，不一致则记录后采用新版本）
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
raw = TASKS_PATH.read_bytes()
print(f'tasks.jsonl bytes={len(raw)} sha256={hashlib.sha256(raw).hexdigest()}')
print(f'HARNESS_README bytes={(DATA_DIR / "HARNESS_README.md").stat().st_size}')
print(f'snapshots={(DATA_DIR / "snapshots").exists()} graphs={(DATA_DIR / "graphs").exists()} embeddings={(DATA_DIR / "embeddings").exists()} wheels={(DATA_DIR / "wheels").exists()}')

from swegemma.models import load_tasks
tasks = load_tasks(TASKS_PATH)
print(f'Loaded {len(tasks)} tasks')

# 基线结构校验：单 Agent、6 工具、无 adapter、无子 Agent 引用
agent_yaml = (AGENT_DIR / 'agent.yaml').read_text(encoding='utf-8')
cfg = yaml.safe_load(agent_yaml)
assert 'adapter' not in cfg, f'003-{VARIANT} 不设置 adapter'
tools = cfg.get('tools', [])
tool_names = [t if isinstance(t, str) else str(t) for t in tools]
assert not any('agent_tool' in t or 'sub_agents' in t for t in tool_names), '003 不用子 Agent'
assert set(tool_names) == {'run_command', 'read_file', 'edit_file', 'write_file', 'get_status', 'submit_patch'}, f'003 只注册6工具，实际={tool_names}'
assert '!include' in agent_yaml and 'prompts/system.md' in agent_yaml, 'instruction 必须 !include prompts/system.md'
print(f'agent tools OK: {tool_names}')

sampling = yaml.safe_load((AGENT_DIR / 'configs' / 'sampling.yaml').read_text(encoding='utf-8'))
assert 'thinking_level' not in str(sampling), '省略 thinking_level'
assert sampling.get('temperature') == 0.2 and sampling.get('top_p') == 0.95, '采样沿用官方 v2 初值'
assert sampling.get('max_output_tokens') == 16384 and sampling.get('thinking_config', {}).get('thinking_budget') == 4096, '输出/思考预算检查'
print('sampling OK')

eval_raw = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8'))
ev = eval_raw.get('evaluation', eval_raw)
for k in ['timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns']:
    assert k in ev, f'eval_config 缺少 {k}（必须显式填写，不依赖默认）'
print(f'eval budgets: {ev}')

# R03：单一有序 manifest；核对指纹/唯一/存在/repo-commit一致/集合互斥/同组不跨集合；未知 scope 报错；禁回退/丢题/首N替换。
SPLITS_PATH = Path('/kaggle/working/task_splits.json')
if not SPLITS_PATH.exists():
    SPLITS_PATH = Path('research/task_splits.json')
assert SPLITS_PATH.exists(), 'R03：缺 task_splits.json 即报错停止，不得回退到 tasks[:3]/[:24]'
splits = json.loads(SPLITS_PATH.read_text(encoding='utf-8'))
print(f'splits from {SPLITS_PATH}: seed={splits["seed"]} sha={splits["source_sha256"]} counts={splits["counts"]}')
assert splits['source_sha256'] == hashlib.sha256(raw).hexdigest(), 'tasks.jsonl 指纹与划分来源不一致：记录差异并形成新数据/划分版本，不得静默换题 (R03)'
by_id = {t.instance_id: t for t in tasks}
assert len(by_id) == len(tasks), '任务 ID 不唯一'
for rid in splits['dev_panel_ids'] + splits['smoke_ids']:
    assert rid in by_id, f'划分 ID 不存在：{rid}（不得静默跳过，R03）'
rec = {r['instance_id']: r for r in splits['records']}
for iid, t in by_id.items():
    if iid in rec:
        assert rec[iid]['repo'] == t.repo and rec[iid]['base_commit'] == t.base_commit, f'repo/commit 不一致：{iid}'
from collections import defaultdict
grp = defaultdict(set)
for r in splits['records']:
    grp[(r['repo'], r['base_commit'])].add(r['split'])
assert all(len(v) == 1 for v in grp.values()), 'R03：同 (repo, base_commit) 组跨集合'
dev_panel = [by_id[i] for i in splits['dev_panel_ids']]
smoke = [by_id[i] for i in splits['smoke_ids']]
assert len(dev_panel) == len(set(splits['dev_panel_ids'])) and len(smoke) == len(set(splits['smoke_ids']))
print(f'dev_panel={len(dev_panel)} smoke={[t.instance_id for t in smoke]}')
print(f'FROZEN smoke: {[t.instance_id for t in smoke]} (启动前固化完整任务 ID 和顺序)')
# 探索集：smoke(splits顺序) + dev_future各1题（种子20260928，不看成功率/参考补丁；见experiment_manifest）
EXTRA_IDS = ['fastapi_15280', 'rich_3905', 'requests_7315']
for _eid in EXTRA_IDS:
    assert _eid in by_id, f'R03：探索ID不存在 {_eid}'
explore = [by_id[i] for i in splits['smoke_ids']] + [by_id[i] for i in EXTRA_IDS]
print(f'FROZEN explore: {[t.instance_id for t in explore]}')

# 003变体一致性：三组采样/预算相同，仅system.md不同 (R06/主计划)
import hashlib as _hl2
for _v in ['a0', 'b', 'c']:
    _base = Path(f'research/003/variants/{_v}/agent')
    assert (_base / 'agent.yaml').exists(), f'缺变体 {_v}'
print('variants a0/b/c present; budgets/sampling frozen across groups (见experiment_manifest)')
print('探索集6题：3 smoke + fastapi_15280/rich_3905/requests_7315；顺序ABC六种排列平衡，冷/热缓存记录')


## 3. 启动模型服务（R05/R06/R07/R11，与001同逻辑，变体无关）


In [ ]:
import importlib.metadata as _md
import time
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
print(f'swegemma dist={_md.version("swegemma")} adk-submission dist={_md.version("adk-submission")} google-adk dist={_md.version("google-adk")}')
t0 = time.time()
declared_model = validate_single_declared_model(AGENT_DIR)
assert declared_model == TARGET_MODEL_NAME, f'R05：声明模型 {declared_model} != 比赛指定 {TARGET_MODEL_NAME}（validate只保唯一，不保等于指定）'
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
assert hasattr(adapters, 'adapters') and isinstance(adapters.adapters, dict), f'R06：未知AdapterManifest形状 type={type(adapters).__name__}；记录版本并停止，不猜测兼容'
n_adapters = len(adapters.adapters)
print(f'adapter_manifest_type={type(adapters).__name__} n_adapters={n_adapters}')
assert n_adapters == 0, f'001 无 LoRA，实际发现 {adapters}'
print(f'declared_model={declared_model} adapters={adapters}')
assert torch.cuda.is_available(), 'R05：无CUDA时在加载前停止（不置gpu_count=1）'
gpu_count = torch.cuda.device_count()
for i in range(gpu_count):
    gp = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {gp.name} mem={gp.total_memory/2**30:.1f}GiB')
print(f'torch={torch.__version__} cuda_build={torch.version.cuda} bf16={torch.cuda.is_bf16_supported()} gpus={gpu_count}')
print(f'model_path={MODEL_PATH} exists={MODEL_PATH.exists()}')
try:
    _mst = MODEL_PATH.stat()
    print(f'model_path stat size={_mst.st_size} mtime={_mst.st_mtime}')
except Exception as e:
    print(f'model_path stat failed: {e}')
for _probe in ['config.json', 'tokenizer.json', 'tokenizer_config.json']:
    print(f'  {_probe} exists={(MODEL_PATH / _probe).exists()}')
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print(f'tp_size={tp_size}（可复现为4xL4；其他卡数/上下文为独立可行性问题，R05）')
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768, dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128, tensor_parallel_size=tp_size, startup_timeout=60*20)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_SVC_LOG = RESULTS_ROOT / 'vllm_server.log'
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
try:
    server_instance.start()
except Exception as e:
    print(f'vLLM start failed: {type(e).__name__}: {e}')
    print(f'startup output={str(getattr(e, "output", ""))[:2000]}')
    print(f'log_path={getattr(server_instance, "log_path", None)}（stop()会删log_path；异常output为空且原日志丢失则记证据缺失，R07）')
    raise
print(f'vLLM started on {server_instance.base_url} tp={tp_size} startup={time.time()-t0:.1f}s')
print(f'server log_path={getattr(server_instance, "log_path", None)} run_archive={RUN_SVC_LOG}')
(models_note, models) = ('registry', server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY'))
print(f'model_registry={models_note} aliases={[declared_model, TARGET_MODEL_NAME]}')
print(f'model_registry_content={str(models)[:500]}')
# R11：保存实际初始prompt与工具schema核对（图目录存在不代表图工具可用；以注册6工具为准）
print(f'tools=6 {[ "run_command","read_file","edit_file","write_file","get_status","submit_patch"]}')
print('注(R11)：litellm.drop_params=True下YAML有字段不等于请求生效；无法从日志核对时标未知，不声称thinking预算已验证。各组同服务同参数路径。')


## 4. 逐题执行（R03/R04/R08/R09/R13；A0/B/C同服务同参数，仓库会话重置，不共享他组答案）


In [ ]:
import asyncio
import concurrent.futures
import hashlib
import json
import sys
import time
import yaml
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

raw_eval = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8'))
sec = raw_eval.get('evaluation', raw_eval)
timeout_seconds = int(sec['timeout_seconds']); max_tool_calls = int(sec['max_tool_calls'])
max_time_minutes = float(sec['max_time_minutes']); max_turns = int(sec['max_turns'])
print(f'budgets: timeout={timeout_seconds}s calls={max_tool_calls} time={max_time_minutes}min turns={max_turns}')

# R03：scope显式枚举，未知报错；禁回退/丢题；启动前已固化ID与顺序，此处只做一致性断言
RUN_SCOPE = 'explore'
assert RUN_SCOPE in ('smoke', 'dev_panel', 'explore'), f'R03：未知RUN_SCOPE={RUN_SCOPE}，停止（不得自动回退/丢题/按首N替换）'
SELECTED = {'smoke': smoke, 'dev_panel': dev_panel, 'explore': explore}[RUN_SCOPE]
assert len(SELECTED) > 0, 'SELECTED为空，中断'
print(f'RUN_SCOPE={RUN_SCOPE} n={len(SELECTED)} FROZEN={[t.instance_id for t in SELECTED]}')

# R04：独立results目录；先写manifest再开始；重试用新attempt+新results_dir（官方文件名只含任务ID）
RESULTS_DIR = RESULTS_ROOT
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(agent_yaml_txt, system_txt) = ((AGENT_DIR / 'agent.yaml').read_text(encoding='utf-8'), (AGENT_DIR / 'prompts' / 'system.md').read_text(encoding='utf-8'))
run_manifest = {
    'run_id': RESULTS_ROOT.name, 'variant': VARIANT, 'run_scope': RUN_SCOPE,
    'task_ids': [t.instance_id for t in SELECTED],
    'agent_yaml_sha256': hashlib.sha256(agent_yaml_txt.encode()).hexdigest(),
    'system_md_sha256': hashlib.sha256(system_txt.encode()).hexdigest(),
    'tasks_sha256': hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest(),
    'eval_budgets': sec, 'sandbox': 'subprocess',
    'model_alias': TARGET_MODEL_NAME, 'declared_model': declared_model,
    'server_base_url': str(getattr(server_instance, 'base_url', '')),
    'outer_start': time.time(),
    'python_version': sys.version.split()[0],
    'executable': sys.executable,
    'gpu_count': globals().get('gpu_count', ''),
    'wheel_manifest': json.loads((WORKING_DIR / 'wheel_manifest_003.json').read_text(encoding='utf-8')) if (WORKING_DIR / 'wheel_manifest_003.json').exists() else 'missing',
}
(RESULTS_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'manifest written: {RESULTS_DIR / "run_manifest.json"}')

limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=RESULTS_DIR, submission_dir=AGENT_DIR, models=models, sandbox='subprocess', timeout_seconds=timeout_seconds, max_time_minutes=max_time_minutes, max_tool_calls=max_tool_calls, max_turns=max_turns, limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters, context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10), events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5), graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels', verbose=False)
evaluator = Evaluator(eval_config)

# R13：公有SwegemmaContext分段计时；R08：逐题append；R09：部分写入与恢复
from swegemma.results import append_task_result
try:
    from swegemma.context import SwegemmaContext as _Ctx
    HAS_CTX = True
except Exception as e:
    _Ctx, HAS_CTX = None, False
    print(f'R13：SwegemmaContext不可用 ({e})；退回有证据日志，不用最后一次get_status当完整时长')
try:
    from swegemma.harness.verification import save_trace_artifact as _save_trace
    HAS_SAVE_TRACE = True
except Exception as e:
    _save_trace, HAS_SAVE_TRACE = None, False
    print(f'R08：save_trace_artifact不可用 ({e})；trace缺失如实记录')

running_results = []
status = {}
for _t in SELECTED:
    status[_t.instance_id] = 'planned'
prep_start = time.time()
for idx, task in enumerate(SELECTED, start=1):
    status[task.instance_id] = 'running'
    print(f'[{idx}/{len(SELECTED)}] {task.instance_id} ({task.repo})...')
    ctx = None
    if HAS_CTX:
        try:
            ctx = _Ctx(docker_manager=evaluator.docker, task=task, graph_dir=eval_config.graph_dir, embeddings_dir=eval_config.embeddings_dir, budget=eval_config.budget, harness=eval_config.harness)
        except Exception as e:
            print(f'  ctx create failed: {type(e).__name__}: {e}')
            ctx = None
    t_outer0 = time.perf_counter()
    try:
        if ctx is not None:
            try:
                r = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED), context=ctx)
            except TypeError:
                print('  context参数不受支持，退回无context调用（记录在案，R13）')
                r = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED))
        else:
            r = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED))
    except KeyboardInterrupt:
        status[task.instance_id] = 'interrupted'
        for _t in SELECTED:
            if status.get(_t.instance_id) == 'planned':
                status[_t.instance_id] = 'not_run'
        print(f'  interrupted；保留现场，不自动无限重试 (R08)')
        break
    except Exception as e:
        status[task.instance_id] = f'error:{type(e).__name__}:{str(e)[:300]}'
        print(f'  error (无TaskResult也逐题记录，R08): {type(e).__name__}: {str(e)[:500]}')
        continue
    t_outer1 = time.perf_counter()
    # R08顺序：先加入累计列表，再调官方持久化
    running_results.append(r)
    try:
        append_task_result(r, RESULTS_DIR, running_results)
    except Exception as e:
        print(f'  append failed: {type(e).__name__}: {str(e)[:500]}')
    # trace补存：append不写trace；对象有trace而文件缺失则补存核对
    try:
        tr_file = RESULTS_DIR / 'traces' / f'trace_{task.instance_id}.json'
        if getattr(r, 'trace', None) is not None and not tr_file.exists() and HAS_SAVE_TRACE:
            _save_trace(r.trace, RESULTS_DIR, task.instance_id)
            print(f'  trace supplemented: {tr_file.exists()}')
        elif getattr(r, 'trace', None) is not None and not tr_file.exists():
            print('  trace object exists but file missing；如实记录缺失 (R08)')
    except Exception as e:
        print(f'  trace supplement failed: {type(e).__name__}: {str(e)[:300]}')
    status[task.instance_id] = 'completed'
    psize = getattr(r, 'agent_patch_size', None)
    if psize is None:
        psize = len(getattr(r, 'agent_patch', None) or '')
    ctx_info = ''
    if ctx is not None:
        _fields = ['task_start_time', 'agent_start_time', 'agent_end_time', 'agent_elapsed_seconds', 'patch_submitted', 'tool_calls', 'total_llm_calls']
        ctx_info = ' ' + ' '.join(f'{_f}={getattr(ctx, _f, "")}' for _f in _fields)
    print(f'  -> resolved={r.resolved} exit={r.test_exit_code} patch_size={psize} calls={r.tool_calls} dur={r.duration_seconds:.1f}s outer={t_outer1-t_outer0:.1f}s{ctx_info}')
(RESULTS_DIR / 'attempt_status.json').write_text(json.dumps(status, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'prep+run wall={(time.time()-prep_start)/60:.1f}min（官方duration含准备+Agent+验证，不含外部模型启动；共同启动成本另列，R13）')
print(f'status: {status}')


## 5. 冻结补丁诊断（R16/R17，仅requests_6592等已开发任务；不注入A0/B/C输入）

R16：skip_agent_patch=True只产空补丁，不能回放；snapshot三路径各归其位，冻结文本走agent_patch，start_time=time.perf_counter()，独立诊断目录，最后清理manager。R17：同时看未修改按预期失败与参考修复通过；空补丁通过/零收集/跳过关键用例不算收益证据。


In [ ]:
import time
from pathlib import Path
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.harness.verification import verify_task

# R16冻结诊断框架（需原补丁文本；缺件则记缺失，不用新运行冒充旧补丁）
DIAG_TASK = 'requests_6592'
DIAG_DIR = RESULTS_ROOT / 'diag_freeze'
DIAG_DIR.mkdir(parents=True, exist_ok=True)
print(f'diag task={DIAG_TASK} dir={DIAG_DIR}')
print('步骤：1)读原Agent补丁记字符数/SHA-256；2)分别timeout=60/300新EvalConfig经Evaluator取同类manager；3)resolve_task_snapshot_paths取三路径；4)verify_task(snapshot三路径归位，patch_path=快照增量，冻结文本走agent_patch，agent_error=None，trace=None，start_time=perf_counter，fast_path语义)；5)独立B沙箱，保存后清理manager，原Agent结束信息仅旁注。')
print('R16注意：直接verify_task的duration/调用数不代表原Agent统计，不与整题时长直接比较。A0新补丁诊断须另名指纹，明确是新实验。')
print('R17：需未修改按预期失败 + 参考修复通过 + 官方resolved/JUnit有效；exit=0不覆盖resolved；不改测试期望/跳过用例。')


## 6. 结果读取（R08/R09；traces补时间线，不替裁判；agent_patch_size为准）


In [ ]:
import json
from pathlib import Path
import pandas as pd

RESULTS_DIR = RESULTS_ROOT
rows = []
tr_path = RESULTS_DIR / 'task_results.jsonl'
bad_lines = []
if tr_path.exists():
    for ln, line in enumerate(tr_path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        try:
            j = json.loads(line)
        except Exception as e:
            bad_lines.append((ln, str(e)[:200], line[:200]))
            continue
        pid = j.get('instance_id', '')
        pfile = RESULTS_DIR / 'patches' / f'{pid}.patch'
        psize = j.get('agent_patch_size')
        ptext = pfile.read_text(encoding='utf-8') if pfile.exists() else ''
        rows.append({
            'instance_id': pid,
            'resolved': j.get('resolved', ''),
            'test_exit_code': j.get('test_exit_code', ''),
            'agent_patch_size': psize,
            'patch_file_chars': len(ptext),
            'patch_nonempty': bool(ptext.strip()),
            'tool_calls': j.get('tool_calls', ''),
            'total_llm_calls': j.get('total_llm_calls', ''),
            'duration_seconds': j.get('duration_seconds', ''),
            'error': j.get('error', ''),
            'test_output_bytes': (RESULTS_DIR / 'test_outputs' / f'{pid}.log').stat().st_size if (RESULTS_DIR / 'test_outputs' / f'{pid}.log').exists() else '',
        })
df = pd.DataFrame(rows)
print(f'rows={len(df)} resolved_sum={(df["resolved"]==True).sum() if len(df) else 0} bad_tail_lines={len(bad_lines)}')
if bad_lines:
    print(f'坏尾行保留为损坏证据，单独报告，不静默丢弃 (R09): {bad_lines[:3]}')
if len(df) == 0:
    print('无数据则报告缺失，不生成看似有效的0分汇总 (R09)')
else:
    display(df.head(30))
    out_csv = RESULTS_DIR / 'analysis_003.csv'
    df.to_csv(out_csv, index=False)
    print(f'saved {out_csv}')
print('traces仅补行为时间线，不替代裁判结果；JSONL无正文不代表空补丁，以patches文件为准 (R08)')
print('按 docs/plan/001 逐题补 primary_cause（人工）：环境/依赖、定位、理解与修改、工具格式或执行、上下文截断、预算、提交失败、尚无法确定')


## 7. 打包（R10；冻结单variant目录；zip指纹；他组/日志/答案在包外）


In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# R10：打包冻结单variant目录；核对全部include/子配置引用/无越界/symlink/单根/官方验证；记录zip指纹
freeze_src = AGENT_DIR
assert (freeze_src / 'agent.yaml').exists(), '冻结目录缺 agent.yaml'
txt = (freeze_src / 'agent.yaml').read_text(encoding='utf-8')
assert 'prompts/system.md' in txt and 'configs/sampling.yaml' in txt, 'R10：全部include目标须存在于包内'
for _ln in txt.splitlines():
    if '!include' in _ln:
        assert '..' not in _ln, f'R10：include越界：{_ln.strip()}'
assert not any((freeze_src / p).is_symlink() for p in ['agent.yaml', 'prompts/system.md', 'configs/sampling.yaml']), 'R10：关键文件不可为symlink'
for sub in sorted(freeze_src.rglob('*')):
    if sub.is_symlink():
        raise AssertionError(f'R10：包内禁symlink：{sub.relative_to(freeze_src)}')
    if sub.is_file() and sub.suffix.lower() not in ('.yaml', '.yml', '.md', ''):
        pass
try:
    from swegemma.models.discovery import validate_single_declared_model as _v
    _decl = _v(str(freeze_src))
    print(f'R10：官方配置验证通过 declared={_decl}')
except Exception as e:
    print(f'R10：官方配置验证异常（记录版本并停止，不猜测兼容）: {type(e).__name__}: {str(e)[:500]}')
    raise
zip_base = RESULTS_ROOT / f'submission-003-{VARIANT}'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=freeze_src))
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    names = [i.filename for i in infos]
    assert 'agent.yaml' in names, f'包根必须有 agent.yaml，实际={names[:10]}'
    assert not any(n.startswith('sub_agents/') or n.startswith('adapters/') for n in names), '001 不带 sub_agents/adapters'
    total = sum(i.file_size for i in infos)
    assert total <= MAX_SUBMISSION_SIZE_BYTES, f'超过3GiB: {total}'
    for info in infos:
        if not info.is_dir():
            assert Path(info.filename).suffix.lower() in ALLOWED_SUBMISSION_EXTENSIONS, f'非法扩展名: {info.filename}'
print(f'Created {zip_path} ({zip_path.stat().st_size/2**20:.2f} MiB) files={len(names)}')
print(names)
print(f'zip sha256={hashlib.sha256(zip_path.read_bytes()).hexdigest()}')
print('R10：Notebook/参考答案/日志/其他候选/训练数据留在包外；不把本地改过的harness打包期待正式评分采用。')


In [ ]:
import shutil
# R07：人工结束单元格——先归档日志再stop（stop会删log_path）；只停本批服务，不无差别杀进程
try:
    lp = getattr(server_instance, 'log_path', None)
    print(f'server log_path={lp}')
    if lp is not None and Path(lp).exists():
        dst = RESULTS_ROOT / 'vllm_server.archived.log'
        shutil.copy(Path(lp), dst)
        print(f'archived server log -> {dst} ({dst.stat().st_size} bytes)')
    else:
        print('无现存server日志文件；若启动异常output也为空则记证据缺失，不冒称已保存 (R07)')
except Exception as e:
    print(f'log archive failed: {type(e).__name__}: {str(e)[:300]}')
try:
    server_instance.stop(timeout=10.0)
    print('server stopped (仅本批次服务)')
except Exception as e:
    print(f'server stop failed, 记录状态不冒称已释放GPU (R07): {type(e).__name__}: {str(e)[:300]}')


## 8. 观察清单（人工回填docs/progress/003-budget-and-context.md）

- 每R编号记改动位置/官方版本/运行证据/未知点；代码改完≠远程已验证。
- 主指标resolved数与配对胜负；验证超时保留分母另列分布。
- 小样本不宣称显著，不外推排行榜。
